# Queue Imbalance: Raw vs. Filtered vs. Shock

Plots the out-of-sample AUC for three versions of queue imbalance, averaged
across all folds in `summary_fixed.csv`, against the null-model baseline of
0.500 (Gould & Bonart 2015, Section 5.4).

| Signal | What it is |
|---|---|
| **raw** | Unfiltered imbalance ratio, straight from observed quantities (Gould & Bonart's own definition) |
| **filtered** | Kalman/IMM-smoothed imbalance - the filter's trend estimate |
| **shock** *(formerly "residual")* | `raw - filtered` - the instantaneous deviation/turbulence in queue imbalance the filter treats as noise and discards. Signed: positive = a sudden burst of buy-side pressure above the recent trend, negative = a sudden burst of sell-side pressure. |

Set `CSV_PATH` below to point at your actual results file.

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

CSV_PATH = "summary_fixed.csv"  # <- change this to your actual output file

HORIZON_ORDER = ["k=10", "k=20", "k=30", "k=50", "k=100"]
NULL_AUC = 0.5

df = pd.read_csv(CSV_PATH)
print(f"Loaded {len(df)} rows from {CSV_PATH}")

n_errors = (df["status"] != "ok").sum() if "status" in df.columns else 0
if n_errors:
    print(f"WARNING: {n_errors} row(s) have status != 'ok' (failed fold/horizon) - excluded below.")
    df = df[df["status"] == "ok"].copy()

df["horizon"] = pd.Categorical(df["horizon"], categories=HORIZON_ORDER, ordered=True)
df = df.sort_values("horizon")

# rename residual -> shock for display, without touching the CSV itself
SIGNAL_COLUMNS = {"raw_auc": "raw", "filtered_auc": "filtered", "residual_auc": "shock"}
print(f"Folds included: {df['fold'].nunique()}  |  Horizons: {df['horizon'].nunique()}")

FileNotFoundError: [Errno 2] No such file or directory: 'summary_fixed.csv'

## Aggregate across folds

Mean AUC per horizon per signal, with standard deviation across folds and the
fraction of folds that beat the null baseline - a single average can hide a
signal that's inconsistent fold-to-fold, so both are shown.

In [ ]:
def summarize(col):
    g = df.groupby("horizon", observed=True)[col]
    return pd.DataFrame({
        "mean_auc": g.mean(),
        "std_auc": g.std(),
        "pct_folds_above_null": g.apply(lambda x: (x > NULL_AUC).mean() * 100),
        "n_folds": g.count(),
    })

summary = {}
for col, label in SIGNAL_COLUMNS.items():
    summary[label] = summarize(col)

for label, s in summary.items():
    print(f"\n=== {label} ===")
    print(s.round(3))

: 

## Plot: mean AUC per horizon, all three signals, vs. the 0.5 null baseline

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5))

colors = {"raw": "#4C72B0", "filtered": "#55A868", "shock": "#C44E52"}
markers = {"raw": "o", "filtered": "s", "shock": "^"}

x = np.arange(len(HORIZON_ORDER))

for label, s in summary.items():
    s = s.reindex(HORIZON_ORDER)
    ax.errorbar(x, s["mean_auc"], yerr=s["std_auc"], label=label,
                color=colors[label], marker=markers[label], markersize=8,
                linewidth=2, capsize=4, capthick=1.5)

ax.axhline(NULL_AUC, color="gray", linestyle="--", linewidth=1.5, label="null baseline (0.500)")

ax.set_xticks(x)
ax.set_xticklabels(HORIZON_ORDER)
ax.set_xlabel("Prediction horizon")
ax.set_ylabel("AUC-ROC (mean across folds, error bars = 1 std)")
ax.set_title("Queue imbalance: raw vs. filtered vs. shock")
ax.legend(loc="best")
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("auc_comparison.png", dpi=150)
plt.show()

## Plot: fraction of folds beating the null baseline

Consistency check - a signal averaging above 0.5 only because a couple of
folds ran hot isn't the same claim as one that beats 0.5 on most folds.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

width = 0.25
for i, (label, s) in enumerate(summary.items()):
    s = s.reindex(HORIZON_ORDER)
    ax.bar(x + (i - 1) * width, s["pct_folds_above_null"], width,
           label=label, color=colors[label])

ax.axhline(50, color="gray", linestyle="--", linewidth=1.5, label="50% of folds (coin flip)")
ax.set_xticks(x)
ax.set_xticklabels(HORIZON_ORDER)
ax.set_xlabel("Prediction horizon")
ax.set_ylabel("% of folds with AUC > 0.5")
ax.set_title("Consistency: how often does each signal beat the null, fold by fold")
ax.legend(loc="best")
ax.grid(alpha=0.3, axis="y")

plt.tight_layout()
plt.savefig("consistency_comparison.png", dpi=150)
plt.show()

## Optional: per-fold spread (boxplot)

Shows the full distribution across folds, not just mean +/- std - useful for
spotting outlier folds or skew a simple std doesn't capture.

In [ ]:
fig, axes = plt.subplots(1, len(HORIZON_ORDER), figsize=(18, 4.5), sharey=True)

for ax, h in zip(axes, HORIZON_ORDER):
    sub = df[df["horizon"] == h]
    data = [sub["raw_auc"], sub["filtered_auc"], sub["residual_auc"]]
    bp = ax.boxplot(data, labels=["raw", "filtered", "shock"], patch_artist=True)
    for patch, label in zip(bp["boxes"], ["raw", "filtered", "shock"]):
        patch.set_facecolor(colors[label])
        patch.set_alpha(0.6)
    ax.axhline(NULL_AUC, color="gray", linestyle="--", linewidth=1)
    ax.set_title(h)
    ax.grid(alpha=0.3, axis="y")

axes[0].set_ylabel("AUC-ROC")
plt.suptitle("Per-fold AUC distribution by horizon", y=1.02)
plt.tight_layout()
plt.savefig("boxplot_by_horizon.png", dpi=150)
plt.show()